# Deploy PolyAI Dialog-RSN-1 on Amazon SageMaker AI and talk to it over HTTP/2

This notebook deploys the PolyAI Dialog-RSN-1 model package from AWS Marketplace to a
SageMaker AI real-time endpoint, asks it one question over a bidirectional HTTP/2 stream,
and then cleans up deployed resources.

### Prerequisites

- Python 3.12 or later. The HTTP/2 client requires it.
- A subscription to the [Dialog-RSN-1 listing on AWS Marketplace](https://aws.amazon.com/marketplace/pp/prodview-l4qyhvfxnkq7e).
  Subscribing is one click and free until an endpoint runs. The code below will not work without this subscription.
- An existing SageMaker AI execution role, and the model package ARN for your Region from the listing.
- Service quota for one `ml.p4d.24xlarge` (or `ml.p5.48xlarge`) endpoint instance in your Region.

> **Cost:** the endpoint is billed while it exists. Run the [Cleanup](#6.-Cleanup) cell when you are done.

## 1. Install dependencies

In [ ]:
%pip install --quiet boto3 "aws-sdk-sagemaker-runtime-http2[awscrt]"

Import boto3 for the SageMaker AI control plane, and the async HTTP/2 runtime client and
event types for the bidirectional stream. `AWSCRTHTTPClient` is the AWS CRT transport
that supports duplex streaming.

In [ ]:
import json
import time

import boto3
from aws_sdk_sagemaker_runtime_http2.client import (
    AsyncSageMakerRuntimeHTTP2Client,
    AsyncSageMakerRuntimeHTTP2Config,
)
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
    ResponseStreamEventInternalStreamFailure,
    ResponseStreamEventModelStreamError,
    ResponseStreamEventPayloadPart,
)
from smithy_http.aio.crt import AWSCRTHTTPClient

## 2. Configuration

In [ ]:
REGION = "us-west-2"  # change region value

EXECUTION_ROLE_ARN = "arn:aws:iam::ACCOUNT_ID:role/SAGEMAKER_EXECUTION_ROLE"

MODEL_PACKAGE_ARN = (
    "arn:aws:sagemaker:REGION:594846645681:model-package/dialog-rsn-1-..."
)

INSTANCE_TYPE = "ml.p4d.24xlarge"  # Or ml.p5.48xlarge

INSTRUCTIONS = (
    "You are the assistant for Maison Lumiere, a restaurant. Answer in one short sentence, "
    "from these facts only: we are open until 11 pm every day, and the nearest parking is "
    "the public car park on Bridge Street, two minutes' walk away."
)

QUESTION = "What time do you close?"

sagemaker = boto3.client("sagemaker", region_name=REGION)

### Helper functions

- `deploy` creates the model, endpoint configuration, and endpoint. Marketplace model
  packages must run with network isolation enabled. It returns once creation is accepted;
  the endpoint is not ready yet.
- `wait_until_ready` polls every 30 seconds until the endpoint is `InService`, and raises
  with the endpoint's `FailureReason` if it fails.
- `delete` removes everything `deploy` created. It is safe to call more than once.

In [ ]:
def deploy(name: str) -> None:
    """Create the model, endpoint config and endpoint. Marketplace packages must run
    network-isolated. Returns once creation is accepted; the endpoint is not ready yet."""
    sagemaker.create_model(
        ModelName=name,
        ExecutionRoleArn=EXECUTION_ROLE_ARN,
        PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
        EnableNetworkIsolation=True,
    )
    sagemaker.create_endpoint_config(
        EndpointConfigName=name,
        ProductionVariants=[
            {
                "VariantName": "AllTraffic",
                "ModelName": name,
                "InitialInstanceCount": 1,
                "InstanceType": INSTANCE_TYPE,
                "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
                "ModelDataDownloadTimeoutInSeconds": 3600,
                "ContainerStartupHealthCheckTimeoutInSeconds": 3600,
            }
        ],
    )
    sagemaker.create_endpoint(EndpointName=name, EndpointConfigName=name)


def wait_until_ready(name: str) -> None:
    """Poll until InService. 20-40 minutes is normal for a p4d pull of a large model.
    Raises with the real FailureReason instead of a bare timeout."""
    while True:
        status = sagemaker.describe_endpoint(EndpointName=name)
        print(f"  {time.strftime('%H:%M:%S')}  {status['EndpointStatus']}")
        if status["EndpointStatus"] == "InService":
            return
        if status["EndpointStatus"] in ("Failed", "RollingBack", "OutOfService"):
            raise RuntimeError(status.get("FailureReason", status["EndpointStatus"]))
        time.sleep(30)


def delete(name: str) -> None:
    """Remove everything deploy() created. Safe to call twice."""
    for call in (
        lambda: sagemaker.delete_endpoint(EndpointName=name),
        lambda: sagemaker.delete_endpoint_config(EndpointConfigName=name),
        lambda: sagemaker.delete_model(ModelName=name),
    ):
        try:
            call()
        except sagemaker.exceptions.ClientError:
            pass

## 3. Deploy the endpoint

In [ ]:
name = f"dialog-rsn-1-{time.strftime('%Y%m%d-%H%M')}"
print(f"deploying {name} on {INSTANCE_TYPE}")
deploy(name)

## 4. Wait for the endpoint to be ready

In [ ]:
try:
    wait_until_ready(name)
except BaseException:
    print(f"deleting {name}")
    delete(name)
    raise

## 5. Ask a question

The endpoint speaks JSON events both ways over a single HTTP/2 connection:

1. `session.update` configures the session: text output and the `INSTRUCTIONS` above.
2. `conversation.item.create` sends the user's question.
3. `response.create` triggers the response.
4. The client reads events until `response.done`, keeping the text from `response.output_text.done`.

`AWSCRTHTTPClient` is required because the default transport cannot do duplex streaming.

In [ ]:
async def ask(name: str, question: str) -> str:
    """Open one conversation, ask one question, return the reply.

    The endpoint speaks JSON events both ways over a single HTTP/2 connection:
    session.update to configure, conversation.item.create for the question,
    response.create to trigger, then read events until response.done.
    AWSCRTHTTPClient is required -- the default transport cannot do duplex streaming.
    """
    config = await AsyncSageMakerRuntimeHTTP2Config.resolve(
        region=REGION, transport=AWSCRTHTTPClient()
    )
    client = AsyncSageMakerRuntimeHTTP2Client(config=config)
    stream = await client.invoke_endpoint_with_bidirectional_stream(
        input=InvokeEndpointWithBidirectionalStreamInput(endpoint_name=name)
    )

    async def send(event: dict) -> None:
        await stream.input_stream.send(
            RequestStreamEventPayloadPart(
                RequestPayloadPart(
                    bytes_=json.dumps(event).encode(),
                    data_type="UTF8",
                    completion_state="COMPLETE",
                )
            )
        )

    async with stream:
        await send(
            {
                "type": "session.update",
                "session": {
                    "type": "realtime",
                    "model": "dialog-rsn-1",
                    "output_modalities": ["text"],
                    "instructions": INSTRUCTIONS,
                    "audio": {
                        "input": {"format": {"type": "audio/pcm", "rate": 16_000}}
                    },
                },
            }
        )
        await send(
            {
                "type": "conversation.item.create",
                "item": {
                    "type": "message",
                    "role": "user",
                    "content": [{"type": "input_text", "text": question}],
                },
            }
        )
        await send({"type": "response.create"})

        _initial, events = await stream.await_output()
        reply = ""
        async for event in events:
            match event:
                case ResponseStreamEventPayloadPart():
                    message = json.loads(event.value.bytes_)
                    if message["type"] == "error":
                        raise RuntimeError(message["error"])
                    if message["type"] == "response.output_text.done":
                        reply = message.get("text", "")
                    elif message["type"] == "response.done":
                        return reply
                case (
                    ResponseStreamEventModelStreamError()
                    | ResponseStreamEventInternalStreamFailure()
                ):
                    raise RuntimeError(event.value.message)
        return reply

In [ ]:
try:
    print("agent:", await ask(name, QUESTION))
except BaseException:
    print(f"deleting {name}")
    delete(name)
    raise

## 6. Cleanup

In [ ]:
print(f"deleting {name}")
delete(name)